## Setup

In [1]:
%%capture
!pip install transformers
!pip install sentencepiece
!pip install seqeval
!pip install datasets

In [2]:
## Mount GDrive
from google.colab import drive
drive.mount('/content/drive/', force_remount=True)

## Imports
import os
import sys
import nltk
import time
import torch
import random
import subprocess
import numpy as np
import pandas as pd
import datetime as dt
from itertools import groupby
from tqdm.notebook import tqdm
from datasets import load_dataset
from transformers import pipeline
from collections import Counter, defaultdict
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForTokenClassification, AutoTokenizer
from seqeval.metrics import f1_score as seq_f1, precision_score as seq_precision, recall_score as seq_recall, classification_report as seq_classification
from sklearn.metrics import f1_score as skl_f1, precision_score as skl_precision, recall_score as skl_recall, classification_report as skl_classification

Mounted at /content/drive/


In [3]:
# Append the library files into the notebook system path for import
sys.path.append('/content/drive/Shareddrives/Machine Translation/Model benchmarking/Libraries/1.0.2')
# import custom library files
import ner, utils

## Load datasets

### Peoples daily
https://huggingface.co/datasets/peoples_daily_ner

In [4]:
pdaily_label_map = {
    "O": 0,
    "B-PER": 1,
    "I-PER": 2,
    "B-ORG": 3,
    "I-ORG": 4,
    "B-LOC": 5,
    "I-LOC": 6,
}

pdaily = ner.ReadNERData()
pdaily_words, pdaily_labels = pdaily.read_dataset('peoples_daily_ner', pdaily_label_map)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:72: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Generating test Split


  0%|          | 0/4637 [00:00<?, ?it/s]

In [5]:
print(ner.check_labels(pdaily_labels))
# Dataset Label Map Alignment to LOC, ORG, PERS, MISC

# harem_label_alignment = {
#     'B-COISA': 'O',
#     'B-LOCAL': 'B-LOC',
#     'B-OBRA': 'O',
#     'I-VALOR': 'O',
#     'I-OUTRO': 'O',
#     'B-ABSTRACCAO': 'O',
#     'B-TEMPO': 'O',
#     'O': 'O',
#     'I-TEMPO': 'O',
#     'B-ACONTECIMENTO': 'O',
#     'I-ORGANIZACAO': 'I-ORG',
#     'I-PESSOA': 'I-PER',
#     'B-PESSOA': 'B-PER',
#     'B-VALOR': 'O',
#     'I-ABSTRACCAO': 'O',
#     'B-ORGANIZACAO': 'B-ORG',
#     'I-COISA': 'O',
#     'I-LOCAL': 'I-LOC',
#     'B-OUTRO': 'O',
#     'I-ACONTECIMENTO': 'O',
#     'I-OBRA': 'O',
# }

# Align the dataset labels to the standard labels
# harem_labels = ner.align_dataset(harem_labels, harem_label_alignment)
# print(ner.check_labels(harem_labels))

{'I-LOC', 'B-PER', 'B-LOC', 'B-ORG', 'I-PER', 'O', 'I-ORG'}


### wikiann

In [6]:
wikiann_label_map = {
    "O": 0,
    "B-PER": 1,
    "I-PER": 2,
    "B-ORG": 3,
    "I-ORG": 4,
    "B-LOC": 5,
    "I-LOC": 6
}

wikiann = ner.ReadNERData()
wikiann_words, wikiann_labels = wikiann.read_dataset('wikiann', wikiann_label_map, lang='zh')

Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/20000 [00:00<?, ? examples/s]

Generating test Split


  0%|          | 0/10000 [00:00<?, ?it/s]

In [ ]:
print(ner.check_labels(wikiann_labels))
# Dataset Label Map Alignment to LOC, ORG, PERS, MISC

{'B-LOC', 'O', 'I-PER', 'B-PER', 'B-ORG', 'I-ORG', 'I-LOC'}


### MRSA
https://huggingface.co/datasets/msra_ner


In [7]:
mrsa_label_map = {
    "O": 0,
    "B-PER": 1,
    "I-PER": 2,
    "B-ORG": 3,
    "I-ORG": 4,
    "B-LOC": 5,
    "I-LOC": 6,
}

mrsa = ner.ReadNERData()
mrsa_words, mrsa_labels = mrsa.read_dataset('msra_ner', mrsa_label_map)

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Generating test Split


  0%|          | 0/3443 [00:00<?, ?it/s]

# Evaluate model

In [8]:
alignment = {
'O'             : 'O',
'B-CARDINAL'    : 'O',
'B-DATE'        : 'O',
'B-EVENT'       : 'O',
'B-FAC'         : 'O',
'B-GPE'         : 'O',
'B-LANGUAGE'    : 'O',
'B-LAW'         : 'O',
'B-LOC'         : 'B-LOC',
'B-MONEY'       : 'O',
'B-NORP'        : 'O',
'B-ORDINAL'     : 'O',
'B-ORG'         : 'B-ORG',
'B-PERCENT'     : 'O',
'B-PERSON'      : 'B-PER',
'B-PRODUCT'     : 'O',
'B-QUANTITY'    : 'O',
'B-TIME'        : 'O',
'B-WORK_OF_ART' : 'O',
'I-CARDINAL'    : 'O',
'I-DATE'        : 'O',
'I-EVENT'       : 'O',
'I-FAC'         : 'O',
'I-GPE'         : 'O',
'I-LANGUAGE'    : 'O',
'I-LAW'         : 'O',
'I-LOC'         : 'I-LOC',
'I-MONEY'       : 'O',
'I-NORP'        : 'O',
'I-ORDINAL'     : 'O',
'I-ORG'         : 'I-ORG',
'I-PERCENT'     : 'O',
'I-PERSON'      : 'I-PER',
'I-PRODUCT'     : 'O',
'I-QUANTITY'    : 'O',
'I-TIME'        : 'O',
'I-WORK_OF_ART' : 'O',
'E-CARDINAL'    : 'O',
'E-DATE'        : 'O',
'E-EVENT'       : 'O',
'E-FAC'         : 'O',
'E-GPE'         : 'O',
'E-LANGUAGE'    : 'O',
'E-LAW'         : 'O',
'E-LOC'         : 'E-LOC',
'E-MONEY'       : 'O',
'E-NORP'        : 'O',
'E-ORDINAL'     : 'O',
'E-ORG'         : 'E-ORG',
'E-PERCENT'     : 'O',
'E-PERSON'      : 'E-PER',
'E-PRODUCT'     : 'O',
'E-QUANTITY'    : 'O',
'E-TIME'        : 'O',
'E-WORK_OF_ART' : 'O',
'S-CARDINAL'    : 'O',
'S-DATE'        : 'O',
'S-EVENT'       : 'O',
'S-FAC'         : 'O',
'S-GPE'         : 'O',
'S-LANGUAGE'    : 'O',
'S-LAW'         : 'O',
'S-LOC'         : 'O',
'S-MONEY'       : 'O',
'S-NORP'        : 'O',
'S-ORDINAL'     : 'O',
'S-ORG'         : 'O',
'S-PERCENT'     : 'O',
'S-PERSON'      : 'O',
'S-PRODUCT'     : 'O',
'S-QUANTITY'    : 'O',
'S-TIME'        : 'O',
'S-WORK_OF_ART' : 'O',
}

model_name = "benjamin/wtp-canine-s-1l"
model_name_output = 'wtp-canine-s-1l'
model_evaluation = ner.ModelEvaluation(
    model_name,
    # alignment
)

config.json:   0%|          | 0.00/6.37k [00:00<?, ?B/s]

KeyError: ignored

In [ ]:
model_evaluation.model.config.id2label

### Peoples daily

In [ ]:
data_name = "peoples_daily_ner"
pdaily_evaluation_output = model_evaluation.evaluate_model(pdaily_words, pdaily_labels)

  0%|          | 0/290 [00:00<?, ?it/s]

/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: Undefin

In [ ]:
pdaily_seqeval = pdaily_evaluation_output.get_classification('Seqeval')
pdaily_seqeval

,Tag,Precision,Recall,F1,support
0,LOC,0.3894,0.0223,0.0421,3637
1,ORG,0.1906,0.3039,0.2343,2185
2,PER,0.4535,0.5735,0.5065,1864
3,micro,0.2999,0.2360,0.2641,7686
4,macro,0.3445,0.2999,0.2610,7686
5,weighted,0.3484,0.2360,0.2094,7686


In [ ]:
pdaily_sklearn = pdaily_evaluation_output.get_classification('Sklearn')
pdaily_sklearn

,Tag,Precision,Recall,F1,support
0,B-LOC,0.8041,0.0327,0.0629,3637
1,B-ORG,0.4532,0.4879,0.4699,2185
2,B-PER,0.6696,0.6470,0.6581,1864
3,E-LOC,0.0000,0.0000,0.0000,0
4,E-ORG,0.0000,0.0000,0.0000,0
5,E-PER,0.0000,0.0000,0.0000,0
6,I-LOC,0.6000,0.0079,0.0157,4918
7,I-ORG,0.6540,0.4411,0.5268,8756
8,I-PER,0.6090,0.4374,0.5091,3601
9,O,0.9431,0.9807,0.9615,193976


### wikiann

In [ ]:
data_name = "wikiann"
wikiann_evaluation_output = model_evaluation.evaluate_model(wikiann_words, wikiann_labels)

  0%|          | 0/625 [00:00<?, ?it/s]

/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: Undefin

In [ ]:
wikiann_seqeval = wikiann_evaluation_output.get_classification('Seqeval')
wikiann_seqeval

,Tag,Precision,Recall,F1,support
0,LOC,0.2028,0.0257,0.0456,4474
1,ORG,0.1244,0.0994,0.1105,4115
2,PER,0.3042,0.6074,0.4053,3943
3,micro,0.2488,0.2329,0.2406,12532
4,macro,0.2104,0.2442,0.1872,12532
5,weighted,0.2089,0.2329,0.1801,12532


In [ ]:
wikiann_sklearn = wikiann_evaluation_output.get_classification('Sklearn')
wikiann_sklearn

,Tag,Precision,Recall,F1,support
0,B-LOC,0.4271,0.0368,0.0678,4371
1,B-ORG,0.3233,0.1868,0.2368,3779
2,B-PER,0.4573,0.7184,0.5589,3899
3,E-LOC,0.0000,0.0000,0.0000,0
4,E-ORG,0.0000,0.0000,0.0000,0
5,E-PER,0.0000,0.0000,0.0000,0
6,I-LOC,0.4684,0.0211,0.0404,12282
7,I-ORG,0.4169,0.1486,0.2191,17399
8,I-PER,0.4480,0.5172,0.4801,12897
9,O,0.8193,0.9038,0.8595,152791


### MRSA

In [ ]:
data_name = "msra_ner"
mrsa_evaluation_output = model_evaluation.evaluate_model(mrsa_words, mrsa_labels)

  0%|          | 0/216 [00:00<?, ?it/s]

/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: Undefin

In [ ]:
mrsa_seqeval = mrsa_evaluation_output.get_classification('Seqeval')
mrsa_seqeval

,Tag,Precision,Recall,F1,support
0,LOC,0.4038,0.0147,0.0284,2852
1,ORG,0.1611,0.3341,0.2174,1320
2,PER,0.3911,0.5186,0.4459,1502
3,micro,0.2611,0.2224,0.2402,5674
4,macro,0.3187,0.2892,0.2306,5674
5,weighted,0.3440,0.2224,0.1829,5674


In [ ]:
mrsa_sklearn = mrsa_evaluation_output.get_classification('Sklearn')
mrsa_sklearn

,Tag,Precision,Recall,F1,support
0,B-LOC,0.7531,0.0214,0.0416,2852
1,B-ORG,0.4153,0.5568,0.4757,1320
2,B-PER,0.6241,0.5925,0.6079,1502
3,E-LOC,0.0000,0.0000,0.0000,0
4,E-ORG,0.0000,0.0000,0.0000,0
5,E-PER,0.0000,0.0000,0.0000,0
6,I-LOC,0.7544,0.0099,0.0195,4363
7,I-ORG,0.5653,0.4380,0.4936,5571
8,I-PER,0.5576,0.4735,0.5121,2925
9,O,0.9465,0.9775,0.9617,151719
